In [ ]:
# T1
!pip install -q language-tool-python

import os, glob, re, json, time, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, ks_2samp

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
asr_dirs = glob.glob('/kaggle/input/**/asr_raw', recursive=True)
print('asr dirs found:', asr_dirs)
ASR_DIR = asr_dirs[0]
WORK = '/kaggle/working'

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
uids = ['train/' + f for f in train.filename] + ['test/' + f for f in test.filename]
print('json files:', len(glob.glob(f'{ASR_DIR}/*.json')), '| expected:', len(uids))

asr = {}
for u in uids:
    with open(f'{ASR_DIR}/{u.replace("/", "__")}.json') as f:
        asr[u] = json.load(f)
print('loaded', len(asr))

In [ ]:
# T2: ASR confidence, timing aur lexical features
FILLERS = {'um', 'umm', 'uh', 'uhh', 'hmm', 'hm', 'mm', 'mmm', 'er', 'erm', 'ah'}

def tok(text):
    return re.findall(r"[a-z']+", text.lower())

def text_of(r):
    return ' '.join(s['text'].strip() for s in r['segments'])

def asr_feats(r):
    segs = r['segments']
    W = [w for s in segs for w in s['words']]
    n = len(W)
    d = {'n_words': n}
    if n == 0 or not segs:
        return d
    dur = r['duration']
    p = np.array([w['p'] for w in W], dtype=float)
    st = np.array([w['s'] for w in W], dtype=float)
    en = np.array([w['e'] for w in W], dtype=float)

    d['wp_mean'], d['wp_median'] = p.mean(), float(np.median(p))
    d['wp_p10'], d['wp_p25'], d['wp_min'] = np.percentile(p, 10), np.percentile(p, 25), p.min()
    for t, nm in ((0.5, '50'), (0.7, '70'), (0.9, '90')):
        d[f'wp_lt{nm}'] = float((p < t).mean())

    lp = np.array([s['avg_logprob'] for s in segs], dtype=float)
    wt = np.array([max(len(s['words']), 1) for s in segs], dtype=float)
    d['seg_logprob_mean'] = float(np.average(lp, weights=wt))
    d['seg_logprob_min'], d['seg_logprob_std'] = lp.min(), lp.std()
    cr = np.array([s['compression_ratio'] for s in segs], dtype=float)
    d['comp_ratio_mean'], d['comp_ratio_max'] = cr.mean(), cr.max()
    d['max_no_speech'] = max(s['no_speech_prob'] for s in segs)

    first, last = st.min(), en.max()
    span = max(last - first, 1e-3)
    d['span'] = span
    d['lead_trail_ratio'] = (first + max(dur - last, 0.0)) / dur
    d['wps_span'] = n / span
    gaps = np.clip(st[1:] - en[:-1], 0, None)
    for t, nm in ((0.25, '250'), (0.5, '500'), (1.0, '1000')):
        d[f'pauses_{nm}_per100'] = 100.0 * float((gaps > t).sum()) / n
    longg = gaps[gaps > 0.25]
    d['pause_time_ratio'] = float(longg.sum()) / span
    d['pause_mean'] = float(longg.mean()) if len(longg) else 0.0
    d['pause_max'] = float(gaps.max()) if len(gaps) else 0.0
    d['artic_rate'] = n / max(span - float(longg.sum()), 1e-3)
    d['word_dur_mean'] = float(np.clip(en - st, 0, None).mean())
    d['segs_per_min'] = len(segs) / (dur / 60.0)
    d['words_per_seg_mean'] = float(np.mean([len(s['words']) for s in segs]))
    d['words_per_seg_std'] = float(np.std([len(s['words']) for s in segs]))
    return d

def mattr(t, window=50):
    if len(t) < window:
        return len(set(t)) / max(len(t), 1)
    return float(np.mean([len(set(t[i:i + window])) / window for i in range(len(t) - window + 1)]))

def lex_feats(text):
    t = tok(text)
    n = max(len(t), 1)
    d = {}
    d['mattr50'] = mattr(t)
    d['avg_word_len'] = float(np.mean([len(w) for w in t])) if t else np.nan
    d['long_word_ratio'] = float(np.mean([len(w) > 6 for w in t])) if t else np.nan
    d['filler_per100'] = 100.0 * sum(w in FILLERS for w in t) / n
    d['rep1_per100'] = 100.0 * sum(t[i] == t[i + 1] for i in range(len(t) - 1)) / n
    bg = list(zip(t[:-1], t[1:]))
    d['rep_bigram_ratio'] = 1.0 - len(set(bg)) / max(len(bg), 1)
    d['sent_end_per100'] = 100.0 * len(re.findall(r'[.!?]', text)) / n
    d['comma_per100'] = 100.0 * text.count(',') / n
    sl = [len(tok(s)) for s in re.split(r'[.!?]+', text) if s.strip()]
    d['sent_len_mean'] = float(np.mean(sl)) if sl else np.nan
    d['sent_len_std'] = float(np.std(sl)) if sl else np.nan
    return d

rows = []
for u in uids:
    d = {'uid': u}
    d.update(asr_feats(asr[u]))
    d.update(lex_feats(text_of(asr[u])))
    rows.append(d)
F_asr = pd.DataFrame(rows).set_index('uid')
print(F_asr.shape)
print(F_asr.isna().sum().sort_values(ascending=False).head(5))

In [ ]:
# T3: LanguageTool
import language_tool_python
tool = language_tool_python.LanguageTool('en-US')

def attr(m, *names):
    for nm in names:
        v = getattr(m, nm, None)
        if v is not None:
            return v
    return None

lt_rows, rule_counts = [], {}
t0 = time.time()
for i, u in enumerate(uids):
    text = text_of(asr[u])
    n = max(len(tok(text)), 1)
    ms = tool.check(text)
    d = {'uid': u, 'lt_total_per100': 100.0 * len(ms) / n}
    cats, rules = Counter(), Counter()
    for m in ms:
        cats[str(attr(m, 'category') or 'NA')] += 1
        rules[str(attr(m, 'ruleId', 'rule_id') or 'NA')] += 1
    for c, v in cats.items():
        d[f'ltcat_{c}_per100'] = 100.0 * v / n
    lt_rows.append(d)
    rule_counts[u] = (rules, n)
    if (i + 1) % 100 == 0:
        print(i + 1, 'done |', round((time.time() - t0) / 60, 1), 'min')
tool.close()

doc_freq = Counter()
for u, (rules, n) in rule_counts.items():
    doc_freq.update(rules.keys())
keep_rules = sorted(r for r, c in doc_freq.items() if c >= 20)
print('rules kept (present in >= 20 files):', len(keep_rules))

lt_df = pd.DataFrame(lt_rows).set_index('uid')
for r in keep_rules:
    lt_df[f'ltrule_{r}_per100'] = [100.0 * rule_counts[u][0].get(r, 0) / rule_counts[u][1] for u in lt_df.index]
lt_df = lt_df.fillna(0.0)
print(lt_df.shape, '| category columns:', [c for c in lt_df.columns if c.startswith('ltcat_')])

In [ ]:
# T4: GPT-2 fluency
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokz = AutoTokenizer.from_pretrained('gpt2')
lm = AutoModelForCausalLM.from_pretrained('gpt2').to(device).eval()

@torch.inference_mode()
def nll_stats(text, prefix):
    ids = tokz(text, return_tensors='pt', truncation=True, max_length=1000)['input_ids'].to(device)
    inp = torch.cat([torch.tensor([[tokz.bos_token_id]], device=device), ids], 1)
    logits = lm(inp).logits[0, :-1]
    nll = torch.nn.functional.cross_entropy(logits.float(), inp[0, 1:], reduction='none').cpu().numpy()
    return {f'{prefix}_nll_mean': float(nll.mean()), f'{prefix}_nll_median': float(np.median(nll)),
            f'{prefix}_nll_p90': float(np.percentile(nll, 90)), f'{prefix}_nll_gt8': float((nll > 8).mean())}

lm_rows = []
t0 = time.time()
for i, u in enumerate(uids):
    text = text_of(asr[u])
    d = {'uid': u}
    d.update(nll_stats(text, 'raw'))
    d.update(nll_stats(' '.join(tok(text)), 'norm'))
    lm_rows.append(d)
    if (i + 1) % 200 == 0:
        print(i + 1, 'done |', round((time.time() - t0) / 60, 1), 'min')
lm_df = pd.DataFrame(lm_rows).set_index('uid')
print(lm_df.shape)

In [ ]:
# T5: combine, save aur univariate analysis
F = F_asr.join(lt_df).join(lm_df)
F.to_csv(f'{WORK}/text_features.csv')
print('saved text_features.csv', F.shape)

D = F.reset_index().merge(train.assign(uid='train/' + train.filename)[['uid', 'label']], on='uid', how='left')
D['split'] = D.uid.str.split('/').str[0]
D['duration'] = [asr[u]['duration'] for u in D.uid]
D['dur_group'] = np.where(D.duration < 40, 'short', np.where(D.duration < 52, 'dur_45s', 'dur_60s'))
trn = D[(D.split == 'train') & (D.label > 0)]
tst = D[D.split == 'test']

def rho(df, c):
    d = df[[c, 'label']].dropna()
    return spearmanr(d[c], d.label)[0] if d[c].nunique() > 2 and len(d) > 20 else np.nan

def ks(c, g):
    a = trn[trn.dur_group == g][c].dropna()
    b = tst[tst.dur_group == g][c].dropna()
    return ks_2samp(a, b).pvalue if len(a) > 5 and len(b) > 5 else np.nan

rows = []
for c in F.columns:
    rows.append(dict(feature=c, rho_all=rho(trn, c),
                     rho_45=rho(trn[trn.dur_group == 'dur_45s'], c),
                     rho_60=rho(trn[trn.dur_group == 'dur_60s'], c),
                     ks_p_45=ks(c, 'dur_45s'), ks_p_60=ks(c, 'dur_60s')))
U = pd.DataFrame(rows).set_index('feature')
U.to_csv(f'{WORK}/univariate_summary.csv')

pd.set_option('display.width', 200)
print('Top 25 by |rho| inside the 45s group (|rho| < 0.16 is within noise for n=154):')
print(U.reindex(U.rho_45.abs().sort_values(ascending=False).index).head(25).round(3).to_string())
print('\nTop 15 by |rho| inside the 60s group:')
print(U.reindex(U.rho_60.abs().sort_values(ascending=False).index).head(15).round(3).to_string())

top = U.reindex(U.rho_45.abs().sort_values(ascending=False).index).head(20)
ax = top[['rho_45', 'rho_60']].plot.barh(figsize=(9, 8))
ax.invert_yaxis()
ax.set_title('Spearman with label: 45s vs 60s group (train, non-zero)')
plt.tight_layout()
plt.show()